# EXP 9: LEX and YACC  — Compiler Design lab
 **Platform:** Google Colab • **Tools:** Flex, Bison, GCC

### Learning outcomes
After this lab, you can identify tokens and lexemes, explain the structure of Lex and YACC files, translate CFG rules into YACC rules, connect a scanner to a parser, and distinguish lexical errors from syntax errors.



## 1. What do LEX and YACC do?
**LEX performs lexical analysis:** it groups characters into tokens. **YACC performs syntax analysis:** it checks whether the token sequence follows a grammar.

For `a=b+c*d;`:

| Lexeme (actual text) | Token category |
|---|---|
| `a` | ID |
| `=` | ASSIGN |
| `b` | ID |
| `+` | PLUS |
| `c` | ID |
| `*` | TIMES |
| `d` | ID |
| `;` | SEMICOLON |

There are **8 tokens**. Whitespace is ignored. `id` in a CFG is a token category; it does not mean that every identifier must literally be spelled `id`.

We use **Flex**, a Lex-style scanner generator, and **GNU Bison**, a YACC-compatible parser generator. GCC compiles their generated C code.

YACC checks whether the **input conforms to the grammar**. It does not determine whether a grammar correctly describes your intended programming language.

## 2. Install the tools
Run this once per Colab runtime. A runtime reset removes installed tools and generated files. `check=True` stops the cell when a command fails.

In [1]:
import subprocess
subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "flex", "bison", "gcc"], check=True)
for tool in ["flex", "bison", "gcc"]:
    print(subprocess.check_output([tool, "--version"], text=True).splitlines()[0])

FileNotFoundError: [WinError 2] The system cannot find the file specified

## 3. Structure of a LEX file
A `.l` file has three sections:
```text
Definitions and C declarations
%%
Patterns and actions
%%
Additional C functions (optional)
```
- `%{ ... %}` encloses C declarations and header includes.
- Each scanner rule has a pattern followed by a C action in `{ ... }`.
- `yytext` contains the text matched by a rule.
- `yylex()` is generated by Flex. Each call scans until a rule returns a token, or end-of-input is reached.
- A rule with no `return` can skip whitespace and continue scanning.
- `yylex()` returns `0` at end-of-input.

Flex selects the **longest match**; if multiple rules match the same length, it selects the rule listed first.

Our identifier pattern is `[a-zA-Z_][a-zA-Z0-9_]*`: a letter or underscore first, followed by zero or more letters, digits, or underscores. Thus `total2` is one ID; `2total` is not a valid identifier under this pattern.

## 4. Structure of a YACC file
A `.y` file also has three sections:
```text
C declarations and YACC declarations
%%
Grammar rules and optional C actions
%%
Supporting C functions
```

| Item | Role |
|---|---|
| `%token ID INVALID` | Declare named token kinds |
| `%start S` | Select the starting nonterminal |
| `%left '+'` | Declare left associativity and a precedence level |
| `%left '*'` | A later declaration gives higher precedence |
| `yyparse()` | Generated parser function; calls `yylex()` |
| `yyerror()` | Our function for reporting parser errors |
| `main()` | Starts parsing and reports the final result |

### Convert the CFG
```yacc
S : E ';' ;
E : E '+' E
  | E '*' E
  | ID
  ;
```
Replace `→` by `:`, use `|` for alternatives, and finish a rule group with `;`. Character terminals are quoted; nonterminals are unquoted. The CFG token `id` becomes the named token `ID`.

In `S : E ';' ;`, the **quoted** `';'` is part of the accepted input. The final **unquoted** `;` ends the YACC rule.

Left recursion can remain in this YACC grammar; it does not need the transformation used for a recursive-descent parser.

## 5. Create `parser.y` for the original grammar
`%%writefile` is a Colab/IPython command: it saves the rest of the cell to a file and must be on the first line. It is not YACC syntax.

The `printf` actions display reductions. The final success message is printed only after `yyparse()` accepts the entire input.

In [ ]:
%%writefile parser.y
%{
#include <stdio.h>
int yylex(void);
void yyerror(const char *message);
%}

%token ID INVALID
%left '+'
%left '*'
%start S

%%

S : E ';' { printf("REDUCE: S -> E ;\n"); }
  ;

E : E '+' E { printf("REDUCE: E -> E + E\n"); }
  | E '*' E { printf("REDUCE: E -> E * E\n"); }
  | ID      { printf("REDUCE: E -> id\n"); }
  ;

%%

void yyerror(const char *message)
{
    printf("PARSER ERROR: %s\n", message);
}

int main(void)
{
    int status = yyparse();
    if (status == 0)
        printf("RESULT: Valid syntax\n");
    else
        printf("RESULT: Invalid syntax\n");
    return status;
}


## 6. Create `lexer.l`
The scanner returns `ID` for an identifier and character codes for operators and the semicolon. These are the same codes used by YACC's grammar.

`parser.tab.h` will be generated by Bison; it shares the definitions of `ID` and `INVALID` with the scanner. Returning `INVALID` ensures an unknown character is rejected rather than silently skipped.

`%option noyywrap` allows this simple scanner to finish at EOF without a user-written `yywrap()` or an additional Flex library. `nounput` and `noinput` omit unused helper functions.

In [ ]:
%%writefile lexer.l
%option noyywrap nounput noinput
%{
#include <stdio.h>
#include "parser.tab.h"
%}

%%

[a-zA-Z_][a-zA-Z0-9_]*  { return ID; }
"="                    { return '='; }
"+"                    { return '+'; }
"*"                    { return '*'; }
";"                    { return ';'; }
[ \t\r\n]+             { /* Ignore whitespace. */ }
.                      { return INVALID; }

%%


## 7. Create a small token-display driver
This C file calls `yylex()` repeatedly and prints every token. It gives us a **separate lexical-analysis stage** before parsing. YACC's parser has its own `main()`, so this driver is compiled into a different executable.

In [ ]:
%%writefile token_driver.c
#include <stdio.h>
#include "parser.tab.h"

int yylex(void);
extern char *yytext;

const char *token_name(int token)
{
    switch (token) {
        case ID:      return "ID";
        case '=':     return "ASSIGN";
        case '+':     return "PLUS";
        case '*':     return "TIMES";
        case ';':     return "SEMICOLON";
        default:      return "INVALID";
    }
}

int main(void)
{
    int token;
    int count = 0;
    int has_error = 0;
    printf("%-15s %s\n", "TOKEN", "LEXEME");
    while ((token = yylex()) != 0) {
        printf("%-15s %s\n", token_name(token), yytext);
        count++;
        if (token == INVALID)
            has_error = 1;
    }
    printf("Items identified: %d\n", count);
    printf("LEXICAL RESULT: %s\n", has_error ? "Invalid character found" : "All tokens recognized");
    return has_error;
}


## 8. Generate and compile
We generate the parser header first because the scanner includes it. We then build two executables:

| Executable | C files linked | Purpose |
|---|---|---|
| `tokens` | `lex.yy.c`, `token_driver.c` | Display all tokens |
| `expression_parser` | `lex.yy.c`, `parser.tab.c` | Check the original CFG |

Generation happens before execution; we will execute the token stage first. No separate token file is needed: during parsing, YACC calls the same scanner directly.

In [ ]:
subprocess.run(["bison", "-Wall", "-d", "parser.y"], check=True)
subprocess.run(["flex", "lexer.l"], check=True)
subprocess.run(["gcc",  "lex.yy.c", "token_driver.c", "-o", "tokens"], check=True)
subprocess.run(["gcc",  "lex.yy.c", "parser.tab.c", "-o", "expression_parser"], check=True)
print("Scanner and expression parser compiled.")

## 9. First use LEX: identify tokens
Python passes the text to the C program's standard input, then closes the stream. This supplies EOF automatically, so the program does not wait for keyboard input in Colab.

In [ ]:
def run_program(program, source):
    result = subprocess.run(
        ["./" + program], input=source, text=True, capture_output=True
    )
    print("INPUT:", repr(source))
    print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    return result.returncode

source = "a=b+c*d;"
lexical_status = run_program("tokens", source)

Expected token output:
```text
TOKEN           LEXEME
ID              a
ASSIGN          =
ID              b
PLUS            +
ID              c
TIMES           *
ID              d
SEMICOLON       ;
Items identified: 8
LEXICAL RESULT: All tokens recognized
```
All characters form valid tokens. That does **not** guarantee that their order is allowed by a grammar.